# Assignment #03

by Patrick Donnelly & Burke Havranek

EECE 5644: Introduction to Machine Learning and Pattern Recognition

Northeastern University College of Engineering

Summer 2026, Session B

## Scenario

### **1. Company Background**

WheelsBazaar is an online used-car marketplace operating across India (similar to Cars24, CarDekho, or
Spinny). Sellers list their vehicles on our platform, and buyers browse, compare, and purchase. We currently
handle roughly 5,000 listings per month across metros and tier-2 cities.

### **2. Business Problem**
Today, sellers set their own asking price, and our operations team manually reviews listings that "look off." This
causes three measurable problems:
1. **Overpriced listings sit unsold.** Cars priced more than ~15% above market value take 3x longer to sell,
clogging inventory and hurting buyer trust in the platform.
2. **Underpriced listings cost sellers money** — and generate complaints when sellers later discover they
sold below market. This damages retention.
3. **Manual review doesn't scale.** Our pricing analysts can review ~40 listings a day. At 5,000
listings/month we review less than 25% of inventory.
We also plan to launch an instant-buy program (WheelsBazaar Direct) where the company purchases cars
directly from sellers. For this, we need an auto

## Part 1: Data Retrieval and Sanitization

In [ ]:
#!/bin/python3
# --Beginning of Code--
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "pandas")

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 80)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

df = pd.read_csv("car_details_v4.csv")
df

Thus, we find 2059 entries with 20 individual fields. First, we examine any missing or malformed entries:

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

We find several across several fields with missing entries. To determine whether they are correlated, let us examine first the most deficient field:

In [ ]:
df[df["Drivetrain"].isnull()].head(10)

From this, we find that the spread of invalid entries tend to correlate quite strongly with each other, but not with any other fields in particular. We are unsure as to the relative importance of each field in determining price, however. Since these incomplete entries compose a rather small percentage of the total data set (~9%), and due to a lack of any defensible default value for the categorical value of `Drivetrain`, we have therefore chosen to drop incomplete entries.

In [ ]:
df.dropna(inplace=True)
df

Let us now consider the format of each entry for each field. For such entries as `Make` and `Model`, these fields are purely categorical, and cannot be meaningfully reduced further other than through encoding, as discussed below. However, for such fields as `Engine`, `Max Power`, and `Max Torque`, these fields carry continuous numerical information in addition to units, and may therefore uniformly sanitized as follows:

In [ ]:
# Given a string, extracts the first floating-point number
get_first_number = lambda x: float(re.search(r'^\d+\.?\d*', x).group())

df["Engine"] = df["Engine"].apply(get_first_number)
df["Max Power"] = df["Max Power"].apply(get_first_number)
df["Max Torque"] = df["Max Torque"].apply(get_first_number)

df

Next, we deal with the categorical data, beginning with the number of unique entries:

In [ ]:
df.nunique()

As expected, the categories of `Fuel Type`, `Transmission`, `Owner`, `Seller Type`, and `Drivetrain` have few entries. For `Color`, `Make`, and especially `Location` and `Model` have significantly more unique entries, the latter most comprising almost half unique entries. It is at this point that we must determine what features are significant to the cost of a vehicle.

While the `Model` of a car certainly affects its price, the sheer number of unique entries relative to the number of total entries makes any model training unlikely to converge upon a pattern, having only on average about two vehicles per unique model. We must therefore drop the `Model` field so as to not introduce undue noise to the model training.

The `Make` of a car, however, is significant to its price, with luxury brands commanding on average higher prices than budget brands. As such, this field must be kept.

The `Location` and `Color` of a car are both interesting pieces of categorical data, as it is unclear whether they might affect the car's final price. For now, we keep them, but may later drop them during model tuning.

In [ ]:
df.drop("Model", axis=1, inplace=True)
df

Let us finally verify the aforementioned categorical data:

In [ ]:
print(df["Make"].unique(),end='\n\n')
print(df["Fuel Type"].unique(),end='\n\n')
print(df["Transmission"].unique(),end='\n\n')
print(df["Location"].unique(),end='\n\n')
print(df["Color"].unique(),end='\n\n')
print(df["Owner"].unique(),end='\n\n')
print(df["Seller Type"].unique(),end='\n\n')
print(df["Drivetrain"].unique(),end='\n\n')

Everything looks correct, with no discernable typos save for `FuelType`, in which there does not appear to be a distinction between `CNG` and `CNG + CNG`, hence the two entries will be consolidated:

In [ ]:
df["Fuel Type"] = df["Fuel Type"].replace("CNG + CNG", "CNG")
df["Fuel Type"].unique()

Lastly, the `Year`, `Transmission`, and `Owner` fields may be made generally numerically rather than categorical, producing `Age`, `Manual`, and `Owners` fields, respectively. Dropping the categorical fields, we produce:

In [ ]:
df["Age"] = 2026 - df["Year"]
df.drop("Year", axis=1, inplace=True)

df["Manual"] = df["Transmission"].map({"Automatic": 1, "Manual": 0})
df.drop("Transmission", axis=1, inplace=True)

df["Owners"] = df["Owner"].map({"UnRegistered Car": 0, "First": 1, "Second": 2, "Third": 3})
df.drop("Owner", axis=1, inplace=True)

df

Categorical data needs to now be encoded for our linear model to meaningfully interpret. For this purpose, we use standard one hot encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cols = ['Fuel Type','Drivetrain','Seller Type','Color','Make','Location']
ohe = OneHotEncoder(drop= 'first', handle_unknown='error', sparse_output=False).set_output(transform='pandas')
ohetransform = ohe.fit_transform(df[cols])
df = pd.concat([df , ohetransform], axis=1).drop(columns = cols)

df.head(10)

As alluded to previously, it is also worth testing if the removal of `Locations` and/or `Color` would result in a better model

In [ ]:
df_no_loc = df.drop(df.filter(regex='Location_.+').columns, axis=1)
df_no_col = df.drop(df.filter(regex='Color_.+').columns, axis=1)
df_no_loc_no_col = df.drop(df.filter(regex='(Location|Color)_.+').columns, axis=1)

df_no_loc_no_col

Continuous data similarly needs to be scaled. In order to affect training bias, we must do so after splitting the data. Due to the several above data sets from which to choose, we create a function for this purpose:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
SCALE_COLS = ["Kilometer", "Engine", "Max Power", "Max Torque",
              "Length", "Width", "Height", "Seating Capacity", "Fuel Tank Capacity", "Age", "Owners"]

def prepare_data(dat):
    X = dat.drop("Price", axis=1)
    Y = dat["Price"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=0xDEADBEEF)
    
    scaler = StandardScaler()
    X_train[SCALE_COLS] = scaler.fit_transform(X_train[SCALE_COLS])
    X_test[SCALE_COLS] = scaler.transform(X_test[SCALE_COLS])
    
    return X_train, X_test, y_train, y_test

## Part 2: Model Creation and Evaluation

It is unknown which model will fare the best. In order to process the several above data sets expeditiously, we again create a helper function for model creation and evaluation:

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

def train_and_test_model(dat, model):
    X_train, X_test, y_train, y_test = prepare_data(dat)
    
    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)
    
    print(f"R^2: {r2_score(y_test, y_hat):0.3f}")
    print(f"MAE: {mean_absolute_error(y_test, y_hat):0.3f}")
    print(f"MAPE: {100*mean_absolute_percentage_error(y_test, y_hat):0.3f}%")

Next, we examine several potential models and parameters:

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

NAMES = ["df", "df_no_loc", "df_no_col", "df_no_loc_no_col"]
ITER = [1, 2, 2, 3]

# Begin with default settings
for i, x in enumerate([df, df_no_loc, df_no_col, df_no_loc_no_col]):
    # High tolerance to prototype solution
    for y in [LinearRegression(tol=1e-3), Lasso(tol=1e-3), Ridge(tol=1e-3)]:
        for j in range(1,ITER[i]+1):
            print(f"{NAMES[i]}: {y.__class__.__name__} (N={j})")
            train_and_test_model(x, make_pipeline(PolynomialFeatures(degree=j), y))

## Part 3: Tuning

From the above pilot test, we indeed find that `df_no_loc_no_col` (that is, the original data with location and color excluded), is generally superior in terms of MAPE, especially with a higher degree polynomial kernel. Given this result, we therefore attempt to tune the corresponding `alpha` parameter for high degree:

In [ ]:
# Default tolerance for tuning
for N in range(2,5):
    # Several orders of magnitude of alpha for comparison
    for a in [0.01, 0.1, 1.0, 10.0, 100.0]:
        print(f"df_no_loc_no_col: Ridge (N={N}, a={a})")
        train_and_test_model(df_no_loc_no_col, make_pipeline(PolynomialFeatures(degree=N), Ridge(alpha=a)))

From this result, we find the lowest MAPE with highest R^2 value near (N=3, a=100). This simultaneously captures both the overall trends in the data set along with minimizing the percentage error per vehicle. Performing some final fine tuning, we find:

In [ ]:
# Fine alpha tuning
for a in [10.0, 25.0, 50.0, 100.0, 250.0, 500.0, 1000.0]:
    print(f"df_no_loc_no_col: Ridge (N=3, a={a})")
    train_and_test_model(df_no_loc_no_col, make_pipeline(PolynomialFeatures(degree=3), Ridge(alpha=a)))

In [ ]:
# Finest alpha tuning
for a in range(50, 150, 10):
    print(f"df_no_loc_no_col: Ridge (N=3, a={a})")
    train_and_test_model(df_no_loc_no_col, make_pipeline(PolynomialFeatures(degree=3), Ridge(alpha=a)))

Part 4: Final Model and Verification

Ergo, we find a minimized model near (N=3, a=60). For practical purposes, we stop here, as increased granularity is unlikely to meaningfully improve our model. Thus, we construct our final model manually for analysis:

In [ ]:
X_train, X_test, y_train, y_test = prepare_data(df_no_loc_no_col)

model = make_pipeline(PolynomialFeatures(degree=3), Ridge(alpha=60, tol=1e-6, max_iter=int(1e5)))
    
model.fit(X_train, y_train)
y_hat = model.predict(X_test)

print(f"R^2: {r2_score(y_test, y_hat):0.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_hat):0.3f}")
print(f"MAPE: {100*mean_absolute_percentage_error(y_test, y_hat):0.3f}%")

# Median as measure of central error
e = y_hat - y_test
APE = abs(e/y_test)
print(f"MedAPE: {100*APE.median():0.3f}%")
print(f"Percentile of 15% error: {100*(sum(APE<=0.15)/len(APE)):0.3f}")

Hence, we show that our model captures 93.4% of the variance of our data, making it extensible, while accurately predicting the fair market rate of 63.7% of cars within 15%. This particular benchmark has been chosen due to the problem outline citing 15% as the approximate limit of user tolerance.